## Measurement of shape and quality descriptors for parts of an object for a given dataset with pixel-level part annotations


### Import modules, set paths and load datasets

In [1]:
import os
import cv2
import json
import numpy as np
import pandas as pd
from tqdm import tqdm
from PIL import Image
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
import scipy
# from scipy.fft import fft

from mahotas.features import zernike_moments
from brisque import BRISQUE # NR-IQA
from pypiqe import piqe # NR-IQA

from skimage import img_as_ubyte
from skimage.measure import regionprops, label, shannon_entropy
from skimage.transform import resize
from skimage.color import rgb2gray
from skimage.filters import laplace
from sklearn.model_selection import train_test_split
from skvideo.measure import niqe # NR-IQA

import torch
from torch import nn
from torchvision import transforms, datasets
from torch.utils.data import DataLoader, Dataset
from torch.optim.lr_scheduler import ReduceLROnPlateau
import timm
import segmentation_models_pytorch as smp

# Patch imresize if missing
if not hasattr(scipy.misc, "imresize"):
    def imresize(arr, size, interp=None, mode=None):
        if isinstance(size, float):  # scale factor
            new_shape = (int(arr.shape[0] * size), int(arr.shape[1] * size))
        else:
            new_shape = size[:2]
        arr_resized = resize(arr, new_shape, order=3, mode="reflect", anti_aliasing=True)
        arr_resized = (arr_resized * 255).astype(np.uint8)
        return arr_resized
    scipy.misc.imresize = imresize

# Patch for deprecated NumPy aliases (for backward compatibility)
if not hasattr(np, 'int'):
    np.int = int
if not hasattr(np, 'float'):
    np.float = float
if not hasattr(np, 'bool'):
    np.bool = bool

Load a dataset with pixel-level part annotations for performing shape analysis

In [2]:
# Hyperparameters
DEVICE_ID = 0
BATCH_SIZE = 128
IMAGE_SIZE = 320

In [3]:
# Create SIFT and ORB detectors once
sift = cv2.SIFT_create()
orb = cv2.ORB_create()
bri_obj = BRISQUE(url=False)

def extract_base_features(mask):
    """Compute geometric, Zernike, Fourier, and texture shape descriptors from a binary mask."""
    
    features = ["area", "perimeter", "aspect_ratio", "extent", "solidity", "eccentricity", 
        "orientation", "circularity", "elongation", "compactness"]
    
    if mask is None or mask.sum() == 0:
        return {f: 0 for f in features}

    # --- Region properties ---
    # mask = mask.astype(np.uint8)
    labeled = label(mask)
    props = regionprops(labeled)
    if len(props) == 0:
        return {f: 0 for f in features}
    p = props[0]
    major_axis = p.major_axis_length
    minor_axis = p.minor_axis_length

    # ----- base shape features -----
    area = p.area
    perimeter = max(p.perimeter, 1e-6) # Ignoring too small perimeters
    aspect_ratio = major_axis / minor_axis if minor_axis > 0 else 0 # L_major / L_minor
    extent = p.extent
    solidity = p.solidity
    eccentricity = p.eccentricity
    orientation = p.orientation
    circularity = 4 * np.pi * area / (perimeter ** 2)
    elongation = 1 - (minor_axis / major_axis) if major_axis > 0 else 0
    # convexity = p.perimeter_convex / perimeter
    compactness = (perimeter ** 2) / (4 * np.pi * area + 1e-6)

    # ----- Assemble features -----
    features_d = {
        "area": area,
        "perimeter": perimeter,
        "aspect_ratio": aspect_ratio,
        "extent": extent,
        "solidity": solidity,
        "eccentricity": eccentricity,
        "orientation": orientation,
        "circularity": circularity,
        "elongation": elongation,
        "compactness": compactness
    }
    return features_d

def compute_sift_features(image, mask=None):
    # if isinstance(image, torch.Tensor):
    #     image = image.detach().cpu().numpy().transpose(1, 2, 0) # Transform tensor to numpy image
    # if isinstance(mask, torch.Tensor):
    #     mask = mask.numpy().astype(np.uint8)
    gray= cv2.cvtColor(img_as_ubyte(image), cv2.COLOR_RGB2GRAY) # converts image into uint8 and mask as input
    keypoints, descriptors = sift.detectAndCompute(gray, mask)
    if descriptors is None:
        descriptors = np.full((0, 128), np.nan, dtype=np.float32)  # return empty array if no keypoints
    return keypoints, descriptors

def compute_orb_features(image, mask=None):
    # if isinstance(image, torch.Tensor):
    #     image = image.detach().cpu().numpy().transpose(1, 2, 0) # Transform tensor to numpy image
    # if isinstance(mask, torch.Tensor):
    #     mask = mask.numpy().astype(np.uint8)
    gray= cv2.cvtColor(img_as_ubyte(image), cv2.COLOR_RGB2GRAY)
    keypoints, descriptors = orb.detectAndCompute(gray, mask)
    if descriptors is None:
        descriptors = np.full((0, 32), np.nan, dtype=np.float32)  # return empty array if no keypoints
    return keypoints, descriptors

def compute_hu_moments(mask):
    # if not isinstance(mask, np.ndarray):
    #     mask = mask.numpy().astype(np.uint8)
    moments = cv2.moments(mask)
    hu = cv2.HuMoments(moments).flatten()
    hu = np.log(np.abs(hu) + 1e-12) # log-scale for stability
    return hu

def compute_zernike_moments(mask, degree=8):
    # if not isinstance(mask, np.ndarray):
    #     mask = mask.numpy().astype(np.uint8)
    radius = min(mask.shape) // 2
    mask_norm = mask / mask.max() if mask.max() > 0 else mask
    zern = zernike_moments(mask_norm, radius=radius, degree=degree)
    return zern

# *** Updated fourier descriptors (Dec 4, 2025)
def compute_fourier_descriptors(mask, image=None, fourier_harmonics=20, visualize=False):
    if not isinstance(mask, np.ndarray): # Ensure proper mask format
        mask = mask.numpy().astype(np.uint8)
    # --- 2. Find largest contour (object part) ---
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
    if not contours:
        return np.full(fourier_harmonics, np.nan, dtype=np.float32)
    cnt = max(contours, key=cv2.contourArea)
    if len(cnt) < 3:
        # Too few points for Fourier transform
        return np.full(fourier_harmonics, np.nan, dtype=np.float32)
    
    # Translation invariance: center contour
    complex_contour = cnt[:,0,0] + 1j * cnt[:,0,1]
    fd = np.fft.fft(complex_contour)
    
    if visualize: # ** IMPORTANT: Visualization uses raw contour (so you can see the real shape), descriptors are centered.
        # Convert image if needed
        H, W = mask.shape
        if image is not None:
            if isinstance(image, torch.Tensor):
                image = image.detach().cpu().numpy().transpose(1, 2, 0)
            elif isinstance(image, Image.Image):
                image = np.array(image.convert('RGB'))
            elif image.dtype != np.uint8:  # NumPy float → uint8
                image = (image*255).astype(np.uint8)
            img_draw = image.copy()
        else:
            img_draw = np.zeros((H, W, 3), dtype=np.uint8)
        cv2.drawContours(img_draw, [cnt.astype(np.int32)], -1, (0, 255, 0), 2)

        fd_recon = fd.copy()
        keep = fourier_harmonics
        if 2 * keep < len(fd_recon):
            fd_recon[keep:-keep] = 0 # Safe zeroing
        else:
            fd_recon[keep:] = 0
        recon = np.fft.ifft(fd_recon)
        pts = np.column_stack((recon.real, recon.imag)).astype(np.int32)

        for i in range(len(pts)):
            cv2.line(img_draw, tuple(pts[i]), tuple(pts[(i + 1) % len(pts)]), (255, 0, 255), 1)
        plt.figure(figsize=(16, 6))
        plt.imshow(img_draw)
        plt.axis('off')
        plt.title("Shape Descriptors Overlay")
        plt.legend(
            handles=[
                Patch(facecolor='green', edgecolor='green'),
                Patch(facecolor='magenta', edgecolor='magenta')
            ],
            labels=["Contour", "Fourier Reconstruction"],
            loc='upper right'
        )
        plt.show()
    
    cnt_centered = complex_contour - np.mean(complex_contour)
    fd = np.fft.fft(cnt_centered)
    if len(fd) < 2 or np.abs(fd[1]) == 0:
        return np.full(fourier_harmonics, np.nan, dtype=np.float32)

    # Scale invariance: divide by first descriptor magnitude
    fd = fd / np.abs(fd[1])

    # Rotation invariance: use only magnitudes
    fd_normalized = np.abs(fd)

    # Keep only first N harmonics
    fd_truncated = fd_normalized[:fourier_harmonics]
    if len(fd_truncated) < fourier_harmonics:
        fd_truncated = np.concatenate([fd_truncated, np.full((fourier_harmonics - len(fd_truncated)), np.nan)])
    return fd_truncated

def extract_shape_features(image, mask):
    # Compute base features
    features = extract_base_features(mask)

    # Compute sift features
    sift_kp, sift_ds = compute_sift_features(image, mask)
    sift_sizes = [k.size for k in sift_kp]
    if sift_ds.shape[0] > 0:
        sift_mean_ds = np.nanmean(sift_ds, axis=0)
    else:
        sift_mean_ds = np.full(128, np.nan)
    sift_dict = {f'sift_ds{i+1}': sift_mean_ds[i] for i in range(len(sift_mean_ds))}
    sift_dict['sift_kp_n'] = len(sift_kp)
    sift_dict['sift_kp_size'] = np.max(sift_sizes) if sift_sizes else 0

    # Compute orb features
    orb_kp, orb_ds = compute_orb_features(image, mask)
    if orb_ds.shape[0] > 0:
        orb_mean_ds = np.nanmean(orb_ds, axis=0)
    else:
        orb_mean_ds = np.full(32, np.nan)
    orb_dict = {f'orb_ds{i+1}': orb_mean_ds[i] for i in range(len(orb_mean_ds))}
    orb_dict['orb_kp_n'] = len(orb_kp)

    # Compute hu moments
    hu_moments = compute_hu_moments(mask)
    hu_dict = {f"hu{i+1}": hu_moments[i] for i in range(len(hu_moments))}

    # Compute Zernike moments
    zern_moments = compute_zernike_moments(mask, degree=8)
    zern_dict = {f"zernike_{i+1}": zern_moments[i] for i in range(len(zern_moments))}

    # Compute fourier descriptors
    fourier_ds = compute_fourier_descriptors(mask, fourier_harmonics=20)
    fourier_dict = {f"fourier_{i+1}": fourier_ds[i] for i in range(len(fourier_ds))}

    features.update(sift_dict)
    features.update(orb_dict)
    features.update(hu_dict)
    features.update(zern_dict)
    features.update(fourier_dict)
    converted = {k: np.float32(v) for k, v in features.items()}
    return converted

def extract_visual_features(image, mask):
    # --- 1. Ensure binary uint8 mask ---
    if not isinstance(mask, np.ndarray):
        mask = mask.numpy().astype(np.uint8)
    # Convert image to numpy
    if isinstance(image, torch.Tensor):
        image = image.detach().cpu().numpy().transpose(1, 2, 0)
    elif isinstance(image, Image.Image):
        image = np.array(image.convert('RGB'))
    img_cropped = np.zeros_like(image)
    img_cropped[mask==1] = image[mask==1]
    # plt.imshow(img_cropped)

    # --- Brightness ---
    brightness = np.mean(img_cropped)

    # --- Contrast (standard deviation of luminance) ---
    gray = rgb2gray(img_cropped)
    contrast = np.std(gray)

    # --- Sharpness (variance of Laplacian) ---
    gray_8u = (gray * 255).astype(np.uint8)
    lap_var = cv2.Laplacian(gray_8u, cv2.CV_64F).var()

    # --- Colorfulness (Hasler & Süsstrunk, 2003) ---
    (R, G, B) = cv2.split(img_cropped)
    rg = np.abs(R - G)
    yb = np.abs(0.5 * (R + G) - B)
    std_rg, std_yb = np.std(rg), np.std(yb)
    mean_rg, mean_yb = np.mean(rg), np.mean(yb)
    colorfulness = np.sqrt(std_rg**2 + std_yb**2) + 0.3 * np.sqrt(mean_rg**2 + mean_yb**2)

    # --- Entropy (texture complexity) ---
    entropy = shannon_entropy(gray)

    # BRISQUE
    bri_obj = BRISQUE(url=False)
    brisque_score = bri_obj.score(img_cropped)

    # NIQE
    niqe_score = niqe(gray)

    # PIQE
    piqe_score, activityMask, noticeableArtifactMask, noiseMask = piqe(gray)

    # --- Aggregate descriptors ---
    descriptors = {
        "brightness": np.float32(brightness),
        "contrast": np.float32(contrast),
        "sharpness": np.float32(lap_var),
        "colorfulness": np.float32(colorfulness),
        "entropy": np.float32(entropy),
        "brisque": np.float32(brisque_score),
        "niqe": np.float32(niqe_score.item()),
        "piqe": np.float32(piqe_score)
    }
    return descriptors

def extract_combined_features(image, mask): 
    # ---- Convert once ----
    if isinstance(image, torch.Tensor):
        image = image.detach().cpu().numpy().transpose(1, 2, 0)
    if isinstance(mask, torch.Tensor):
        mask = mask.detach().cpu().numpy().astype(np.uint8)
    mask_u8 = mask.astype(np.uint8)
    image_f = (image - image.min()) / (image.max() - image.min() + 1e-8) # Linearly rescale to [0, 1] and avoid division by zero
    image_u8 = img_as_ubyte(image_f)

    combined_features = extract_shape_features(image_u8, mask_u8)
    vis_features = extract_visual_features(image_f, mask_u8)
    combined_features.update(vis_features)
    return combined_features

def extract_all_features(image, mask):
    abdomen_mask = mask == 1
    head_mask = mask == 2
    thorax_mask = mask == 3
    full_mask = mask > 0

    head_feats = extract_combined_features(image, head_mask)
    thorax_feats = extract_combined_features(image, thorax_mask)
    abdomen_feats = extract_combined_features(image, abdomen_mask)
    body_feats = extract_combined_features(image, full_mask)

    # Ratios
    area_sum = head_feats["area"] + thorax_feats["area"] + abdomen_feats["area"]
    ratios = {
        "head_to_thorax_area": head_feats["area"] / (thorax_feats["area"] + 1e-6),
        "thorax_to_abdomen_area": thorax_feats["area"] / (abdomen_feats["area"] + 1e-6),
        "head_to_total_area": head_feats["area"] / (area_sum + 1e-6),
        "thorax_to_total_area": thorax_feats["area"] / (area_sum + 1e-6),
        "abdomen_to_total_area": abdomen_feats["area"] / (area_sum + 1e-6),
    }

    record = {}
    record.update({f"head_{k}": v for k, v in head_feats.items()})
    record.update({f"thorax_{k}": v for k, v in thorax_feats.items()})
    record.update({f"abdomen_{k}": v for k, v in abdomen_feats.items()})
    record.update({f"full_{k}": v for k, v in body_feats.items()})
    record.update(ratios)
    return record

As discussed, each part mask has 225 shape descriptors and 8 visual descriptors. We have five values for inter-part ratios: head_to_thorax, thorax_to_abdomen, head_to_total, thorax_to_total and abdomen_to_total. Therefore, total number of descriptors for all parts (head, thorax, abdomen, fullbody) = (225+8)*4 + 5 = 932 + 5 = 937

In [ ]:
def get_smp_preprocess(image_size):
    return transforms.Compose([
        transforms.Resize((image_size, image_size)),
        transforms.ToTensor(),
        # ⚠️ Use SAME normalization as training
        # transforms.Normalize(mean=..., std=...)
    ])

@torch.no_grad()
def predict_mask(model, image, device="cuda"):
    """
    image: torch.Tensor [3, H, W] in [0,1]
    return: np.ndarray [H, W] with class labels
    """
    if image.dim() == 3:
        image = image.unsqueeze(0)

    image = image.to(device)
    logits = model(image)                 # [B, C, H, W]
    pred = torch.argmax(logits, dim=1)    # [B, H, W]
    return pred.squeeze(0).cpu().numpy().astype(np.uint8)

class ImageOnlyDataset(Dataset):
    def __init__(self, root, images_dir="images", image_size=320):
        self.images_dir = os.path.join(root, images_dir)
        self.image_paths = sorted([
            os.path.join(self.images_dir, f)
            for f in os.listdir(self.images_dir)
            if f.lower().endswith((".png", ".jpg", ".jpeg"))
        ])
        self.transform = get_smp_preprocess(image_size)

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        path = self.image_paths[idx]
        img = Image.open(path).convert("RGB")
        img = self.transform(img)
        return img, path

In [ ]:
# Load the datasets
home_path = r"/home/c/choton/beemachine/datasets/Beemachine_Partwhole_v1/"
train_path = os.path.join(home_path, "train") # Path for the training data
val_path = os.path.join(home_path, "valid") # Path for validation data
test_path = os.path.join(home_path, "test") # Path for testing data

train_dataset = ImageOnlyDataset(root=train_path, image_size=IMAGE_SIZE)
val_dataset = ImageOnlyDataset(root=val_path, image_size=IMAGE_SIZE)
test_dataset = ImageOnlyDataset(root=test_path, image_size=IMAGE_SIZE)

In [ ]:
def extract_features_with_model( dataset, model, output_csv, device="cuda"):
    records = []
    print(f"Extracting features for {len(dataset)} images using segmentation model...")

    for image, img_path in tqdm(dataset):
        # 1. Predict segmentation
        pred_mask = predict_mask(model, image, device=device)
        # 2. Extract features (unchanged code)
        features = extract_all_features(image, pred_mask)
        record = {"image": os.path.basename(img_path)}
        record.update(features)
        records.append(record)

    df = pd.DataFrame(records)
    df.to_csv(output_csv, index=False)
    print(f"✅ Saved features to: {output_csv}")
    return df


In [6]:
def extract_features_for_dataset(dataset, output_csv):
    records = []
    print(f"Extracting shape features for {len(dataset)} images...")
    for image, mask, img_path in tqdm(dataset):
        features = extract_all_features(image, mask)
        record = {"image": os.path.basename(img_path)}
        record.update(features)
        records.append(record)

    df = pd.DataFrame(records)
    df.to_csv(output_csv, index=False)
    print(f"✅ Saved shape features to: {output_csv}")
    return df

In [7]:
feat_path = r"./bee_gt_shape_features_predicted"
os.makedirs(feat_path, exist_ok=True)

In [8]:
train_csv = os.path.join(feat_path, r"bee_predicted_features_train.csv")
df = extract_features_for_dataset(train_dataset, train_csv)
df

Extracting shape features for 23903 images...


  0%|          | 17/23903 [00:14<5:34:39,  1.19it/s]/home/c/choton/miniconda3/envs/beemachine_new/lib/python3.11/site-packages/brisque/brisque.py:114: RuntimeWarning: invalid value encountered in scalar divide
  return (np.sum(np.abs(x)) / size) ** 2 / (np.sum(x ** 2) / size)
/home/c/choton/miniconda3/envs/beemachine_new/lib/python3.11/site-packages/brisque/brisque.py:124: RuntimeWarning: invalid value encountered in divide
  return squares_sum / ((filtered_values.shape))
/home/c/choton/miniconda3/envs/beemachine_new/lib/python3.11/site-packages/pypiqe/piqe.py:142: RuntimeWarning: invalid value encountered in divide
  ipImage = np.round(255 * (ipImage / np.max(ipImage)))
100%|██████████| 23903/23903 [6:22:39<00:00,  1.04it/s]    


✅ Saved shape features to: ./bee_gt_shape_features_concise/bee_gt_features_train.csv


,image,head_area,head_perimeter,head_aspect_ratio,head_extent,head_solidity,head_eccentricity,head_orientation,head_circularity,head_elongation,...,full_colorfulness,full_entropy,full_brisque,full_niqe,full_piqe,head_to_thorax_area,thorax_to_abdomen_area,head_to_total_area,thorax_to_total_area,abdomen_to_total_area
0,00903Q80CQ70TQX0AR403QU0Q050BRZQTQJKWR70JQN0YQ...,5372.0,435.782806,1.470436,0.500000,0.739741,0.733147,1.247312,0.355472,0.319929,...,0.153390,6.543673,58.926941,23.382116,46.903793,0.179882,16.646601,0.145072,0.806481,0.048447
1,00903Q80CQ70TQX0AR403QU0Q050BRZQTQJKWR70JQN0YQ...,50.0,30.485281,2.958705,0.714286,0.892857,0.941151,-1.431646,0.676082,0.662014,...,0.153336,6.540353,58.953335,28.662790,44.726948,0.001674,32.601528,0.001622,0.968666,0.029712
2,00903Q80CQ70TQX0AR403QU0Q050BRZQTQJKWR70JQN0YQ...,5363.0,434.611206,1.474576,0.499162,0.740746,0.734913,-1.245671,0.356793,0.321839,...,0.153380,6.541152,70.804543,29.400019,45.492737,0.179587,16.646042,0.144868,0.806672,0.048460
3,00903Q80CQ70TQX0AR403QU0Q050BRZQTQJKWR70JQN0YQ...,50.0,30.485281,2.958705,0.714286,0.892857,0.941151,1.431646,0.676082,0.662014,...,0.153313,6.542576,70.929466,25.411312,45.126392,0.001674,32.602619,0.001622,0.968667,0.029711
4,00903Q80CQ70TQX0AR403QU0Q050BRZQTQJKWR70JQN0YQ...,52.0,30.278175,3.072477,0.577778,0.852459,0.945552,-0.159826,0.712777,0.674530,...,0.153419,6.541544,60.084309,23.681786,46.188679,0.001746,32.378262,0.001691,0.968401,0.029909
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
23898,YKBRFKBR3K1RFK1R50CQLQTQFKVRX0VR40TQI0Z060FRW0...,13838.0,746.500610,1.490742,0.534368,0.825509,0.741632,0.690362,0.312049,0.329193,...,0.113217,8.655890,42.128090,28.435749,41.316868,0.431198,5.298333,0.266182,0.617308,0.116510
23899,YKBRFKBR3K1RFK1R50CQLQTQFKVRX0VR40TQI0Z060FRW0...,13838.0,746.500610,1.490742,0.534368,0.825509,0.741632,-0.690362,0.312049,0.329193,...,0.112703,8.605858,54.530071,27.508842,45.956921,0.431198,5.298333,0.266182,0.617308,0.116510
23900,YKBRFKBR3K1RFK1R50CQLQTQFKVRX0VR40TQI0Z060FRW0...,13838.0,746.500610,1.490742,0.534368,0.825509,0.741632,-0.690362,0.312049,0.329193,...,0.113192,8.696109,54.062828,28.378418,42.370674,0.431198,5.298333,0.266182,0.617308,0.116510
23901,YKBRFKBR3K1RFK1R50CQLQTQFKVRX0VR40TQI0Z060FRW0...,13777.0,743.707703,1.478657,0.532013,0.822802,0.736636,0.879852,0.313011,0.323711,...,0.112702,8.635025,39.072590,27.127497,44.518559,0.430128,5.272428,0.265545,0.617362,0.117093


In [9]:
val_csv = os.path.join(feat_path, r"bee_gt_features_val.csv")
df = extract_features_for_dataset(val_dataset, val_csv)
df

Extracting shape features for 2812 images...


  0%|          | 0/2812 [00:00<?, ?it/s]/home/c/choton/miniconda3/envs/beemachine_new/lib/python3.11/site-packages/brisque/brisque.py:114: RuntimeWarning: invalid value encountered in scalar divide
  return (np.sum(np.abs(x)) / size) ** 2 / (np.sum(x ** 2) / size)
/home/c/choton/miniconda3/envs/beemachine_new/lib/python3.11/site-packages/brisque/brisque.py:124: RuntimeWarning: invalid value encountered in divide
  return squares_sum / ((filtered_values.shape))
/home/c/choton/miniconda3/envs/beemachine_new/lib/python3.11/site-packages/pypiqe/piqe.py:142: RuntimeWarning: invalid value encountered in divide
  ipImage = np.round(255 * (ipImage / np.max(ipImage)))
100%|██████████| 2812/2812 [41:03<00:00,  1.14it/s]


✅ Saved shape features to: ./bee_gt_shape_features_concise/bee_gt_features_val.csv


,image,head_area,head_perimeter,head_aspect_ratio,head_extent,head_solidity,head_eccentricity,head_orientation,head_circularity,head_elongation,...,full_colorfulness,full_entropy,full_brisque,full_niqe,full_piqe,head_to_thorax_area,thorax_to_abdomen_area,head_to_total_area,thorax_to_total_area,abdomen_to_total_area
0,0090L0W0TQIQUR0QCR0QYRZQJR7QK060Z0M0VRIQTRE0TR...,37092.0,2099.795166,1.240526,0.415122,0.580288,0.591766,0.231031,0.105715,0.193891,...,0.158385,9.824629,47.759716,24.217192,34.562733,3.709200e+10,0.000000,0.638043,0.000000,0.361957
1,0090L0W0TQIQURHQ3R7QDR3KBRHQDQ50K0KQTRYKCQ70JQ...,3582.0,515.380798,1.984778,0.324163,0.519658,0.863800,-0.246266,0.169465,0.496165,...,0.141668,8.502266,53.944126,23.956165,40.755508,1.022435e-01,2.788887,0.069991,0.684552,0.245457
2,0090L0W0TQIQURJK1RLQVRFKUR3KWRFKTQM0JQG01RLQOQ...,3348.0,386.959412,1.351244,0.492353,0.770718,0.672542,1.488085,0.280973,0.259941,...,0.174188,6.604094,51.412582,26.019033,65.370392,1.315624e-01,2.624858,0.086981,0.661142,0.251877
3,0090L0W0TQIQURLQUR3K9RZQVRHQ3RW0OR20Q020Z0M0VR...,2047.0,338.492432,2.714557,0.278503,0.554893,0.929674,0.910195,0.224507,0.631616,...,0.192691,8.739911,41.483814,22.852491,37.664986,5.255726e-02,3.840268,0.040030,0.761640,0.198330
4,0090L0W0TQIQURSQFR7QTR0Q3R7Q3RU0YR90CR7Q000QL0...,2163.0,434.102600,1.361810,0.275401,0.364387,0.678807,0.271949,0.144239,0.265683,...,0.059882,3.789318,79.247589,24.520424,71.076874,1.091873e-01,3.003791,0.075714,0.693433,0.230853
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2807,Bombus_dahlbomii_GBIF_iNat_1847588502_1-jpg_jp...,3827.0,578.688354,1.431781,0.192447,0.361515,0.715678,-1.149381,0.143608,0.301569,...,0.230743,5.101147,75.699081,26.075346,40.255268,2.186857e+00,0.177719,0.248120,0.113460,0.638421
2808,Bombus_dahlbomii_GBIF_iNat_1898802975_1-jpg_jp...,2428.0,635.031555,3.328330,0.133671,0.234953,0.953797,-0.094259,0.075660,0.699549,...,0.198478,6.194665,95.124741,27.007528,87.206230,1.549853e-01,0.814072,0.065028,0.419573,0.515400
2809,Bombus_dahlbomii_GBIF_iNat_1898803109_1-jpg_jp...,3076.0,365.048767,2.692707,0.300567,0.586911,0.928484,-1.108631,0.290064,0.628626,...,0.177723,6.674338,51.284622,24.726168,42.927013,7.289100e+00,0.025736,0.154612,0.021211,0.824177
2810,Bombus_zonatus_GBIF_iNat_2236827425_2-jpg_jpg....,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,NaN,27.075386,100.000000,0.000000e+00,0.000000,0.000000,0.000000,0.000000


In [10]:
test_csv = os.path.join(feat_path, r"bee_gt_features_test.csv")
df = extract_features_for_dataset(test_dataset, test_csv)
df

Extracting shape features for 1407 images...


  0%|          | 1/1407 [00:00<22:42,  1.03it/s]/home/c/choton/miniconda3/envs/beemachine_new/lib/python3.11/site-packages/brisque/brisque.py:114: RuntimeWarning: invalid value encountered in scalar divide
  return (np.sum(np.abs(x)) / size) ** 2 / (np.sum(x ** 2) / size)
/home/c/choton/miniconda3/envs/beemachine_new/lib/python3.11/site-packages/brisque/brisque.py:124: RuntimeWarning: invalid value encountered in divide
  return squares_sum / ((filtered_values.shape))
/home/c/choton/miniconda3/envs/beemachine_new/lib/python3.11/site-packages/pypiqe/piqe.py:142: RuntimeWarning: invalid value encountered in divide
  ipImage = np.round(255 * (ipImage / np.max(ipImage)))
100%|██████████| 1407/1407 [21:24<00:00,  1.10it/s]


✅ Saved shape features to: ./bee_gt_shape_features_concise/bee_gt_features_test.csv


,image,head_area,head_perimeter,head_aspect_ratio,head_extent,head_solidity,head_eccentricity,head_orientation,head_circularity,head_elongation,...,full_colorfulness,full_entropy,full_brisque,full_niqe,full_piqe,head_to_thorax_area,thorax_to_abdomen_area,head_to_total_area,thorax_to_total_area,abdomen_to_total_area
0,0090L0W0TQIQUR0Q3RIQCRIQTRQQS00QL0P0OQLQJRSQ9R...,4238.0,604.546265,3.549748,0.246740,0.560805,0.959500,1.296008,0.145718,0.718290,...,0.129201,6.360455,61.413651,24.600269,43.924515,1.405126e-01,14.479597,0.116167,0.826736,0.057097
1,0090L0W0TQIQUR0QCR0QYRZQJR7QK060Z0M0VRIQTRE0TR...,37008.0,2079.238770,1.243384,0.414182,0.580326,0.594282,1.341561,0.107571,0.195743,...,0.158416,9.805406,42.587055,24.086582,32.898232,3.700800e+10,0.000000,0.637926,0.000000,0.362074
2,0090L0W0TQIQURHQ3RXQDRSQCRP0H060Z0IQDRMQORMQJR...,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.048678,5.311530,79.578094,28.716291,60.758263,0.000000e+00,3.482295,0.000000,0.776900,0.223100
3,0090L0W0TQIQURJK9RFKVR3KVRIQJRKQTR20JR90H0E0TQ...,7123.0,526.215271,1.182975,0.522789,0.737676,0.534250,-0.402206,0.323255,0.154674,...,0.081855,8.399035,63.823784,25.243011,56.811680,1.675093e-01,98.432869,0.142238,0.849135,0.008627
4,0090L0W0TQIQURJKBRJKNRKQCRSQTR7QJR20TRIQZ0KQFR...,4810.0,602.564514,1.377386,0.243421,0.389348,0.687681,1.143645,0.166475,0.273987,...,0.152707,6.471943,91.050224,25.726192,67.698029,1.895044e-01,28.583334,0.154762,0.816667,0.028571
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1402,Bombus_dahlbomii_GBIF_iNat_1831206830_1-jpg_jp...,11541.0,656.564514,1.501302,0.578728,0.806217,0.745873,-0.182132,0.336433,0.333912,...,0.296208,8.928906,38.434128,23.377199,40.812935,8.866779e-01,0.522878,0.233387,0.263215,0.503397
1403,Bombus_dahlbomii_GBIF_iNat_1847588502_1-jpg_jp...,3827.0,578.688354,1.431781,0.192447,0.361515,0.715678,-1.149381,0.143608,0.301569,...,0.230815,5.101389,75.682793,25.436483,42.560959,2.716689e-01,1.430588,0.137855,0.507438,0.354706
1404,Bombus_zonatus_GBIF_iNat_2236827425_1-jpg_jpg....,9326.0,697.824402,2.227183,0.407320,0.585400,0.893533,-1.510278,0.240665,0.551002,...,0.242092,6.379587,49.145760,24.052692,32.615612,7.733002e-01,0.758395,0.250107,0.323428,0.426464
1405,Bombus_zonatus_GBIF_iNat_2236827425_1-jpg_jpg....,9350.0,700.067078,2.229703,0.404552,0.582373,0.893788,0.064913,0.239741,0.551510,...,0.242333,6.387228,50.890129,24.677080,31.672842,7.747121e-01,0.758341,0.250442,0.323271,0.426287
